In [1]:
import sys, os
sys.path.append(os.path.abspath('../../'))
import setup_path

In [2]:
import numpy as np
import pandas
from collections import defaultdict
import random
import time

from docplex.mp.model import Model
import cplex

from src.problems.MCLP import get_neighbors, MCLP
from src.utils.cplex_helpers import solution, cont_solution, check_feasibility, WS_parameters
from src.utils.OQUBO import DocplexModeltoQUBO, QUBOtoIsingModel
from src.utils.helpers import compute_expect, compute_cost, BruteForceSolutions

import datetime
import yfinance as yf

import qiskit
from qiskit_aer import AerSimulator
from qiskit.quantum_info import SparsePauliOp
from scipy.optimize import minimize
from qiskit import QuantumCircuit, transpile
from qiskit.circuit import Parameter
from qiskit.quantum_info import Statevector, DensityMatrix, Pauli
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

from qiskit.circuit.library import efficient_su2
from qiskit.circuit import ParameterVector

In [3]:
def min_QAOA(model, ising_model, p, backend, LR=False, WS_params=None, iters=500):
    enrgs = []
    probs_opt = []
    def Cost_function(params, ising_model, model, quantum_circuit, backend, p, gammas, betas, LR=False, minimize=True):
        global n_iters
        
        if LR:
            gamma_vals = np.arange(1, p + 1) * params[0]/p # Annealing liear ramp schecule (cost hamiltonian)
            beta_vals = np.arange(1, p + 1)[::-1] * params[1]/p # Annealing liear ramp schedule (mixer term)
        else:
            gamma_vals = params[:p]
            beta_vals =  params[p:]
        
        bind_dict = {gammas[i]: gamma_vals[i] for i in range(len(gamma_vals))}
        bind_dict.update({betas[i]: beta_vals[i]   for i in range(len(beta_vals))})
        qc_exec = quantum_circuit.assign_parameters(bind_dict)
    
        shots = 1000
        result = backend.run(qc_exec, shots=shots).result().get_counts() 
    
        opt_sol = solution(model)
        prob_opt = result.get(opt_sol,0)/shots
        energy_qubo = compute_cost(result, ising_model)
        #print(f'Energy reached: {energy_qubo}, Probability (Opt_sol): {prob_opt}')
        enrgs.append(energy_qubo)
        probs_opt.append(prob_opt)
        if minimize:
            return energy_qubo
        else: 
            return energy_qubo, result
    
    n_qubits = model.number_of_variables
    gammas = ParameterVector('γ', length=p)
    betas = ParameterVector('β', length=p)
    
    hamiltonian, ising_constant = ising_model
    
    ansatz = QuantumCircuit(n_qubits)

    if WS_params:
        for i, param in enumerate(WS_params):
            ansatz.ry(param, i)
    else:
        ansatz.h(range(n_qubits))
        
    for ii in range(p):
        for qbits, value in hamiltonian.items():
            if qbits[0] == qbits[1]:
                ansatz.rz(2*gammas[ii]*float(value), qbits[0])
        for qbits, value in hamiltonian.items():
            if qbits[0] != qbits[1]:
                ansatz.rzz(2*gammas[ii]*float(value), *qbits)
        if WS_params:
            for i, param in enumerate(WS_params):
                ansatz.ry(-param, i)
            ansatz.rz(-2*betas[ii], range(n_qubits))
            for i, param in enumerate(WS_params):
                ansatz.ry(param, i)
        else:
            ansatz.rx(-2*betas[ii], range(n_qubits))
            
    ansatz = ansatz.reverse_bits()
    ansatz.measure_all()
    gates = dict(ansatz.count_ops())
    
    # Transpile the ansatz circuit for the given backend
    qc = transpile(ansatz, backend=backend)
    
    if LR:
        delta_gamma = np.random.random() 
        delta_beta = np.random.random() 
        params = [delta_gamma, delta_beta]
        bounds_params = [(0.0000001, 0.9), (0.0000001, 0.9)] 
    else:
        gammas0 = np.random.normal(0, 1, size=p)
        betas0 = np.random.normal(0, 1, size=p)
        params = np.concatenate([gammas0, betas0])
        bounds_params = [(0.0000001, np.pi)] * (2*p)

    res = minimize(Cost_function,
            params,
            args=(ising_model, model, qc, backend, p, gammas, betas, LR),
            bounds=bounds_params,
            method="cobyla",
            options={"maxiter": iters})

    params = res.x
    f_energy, qaoa_result = Cost_function(params, ising_model, model, qc, backend, p, gammas, betas, LR, minimize=False)
    
    return res, enrgs, probs_opt, qaoa_result, f_energy, gates

In [4]:
with open("data_MCLP_.txt", "r") as file:
    instances_dict = eval(file.read())

In [5]:
def hammingDist(str1, str2):
    i = 0
    count = 0

    while(i < len(str1)):
        if(str1[i] != str2[i]):
            count += 1
        i += 1
    return count

In [6]:
def run_MCLP(grid_size, instances_dict, ps, N, alpha, EqC=2, slack_vars=False, WS=False, LR=False, iters=500):
    shots = 1000

    data = {}
    data['r'] = []
    data['probs'] = []
    data['hamming_d'] = []
    data['qaoa_FR'] = []
    data['gates'] = []
    data['t_qaoa'] = []
    data['t_total'] = []
    data['n_fev'] = []
    data['n_qubits'] = []
    data['ham_energies'] = []
    data['qaoa_e'] = []
    
    print(f'{grid_size} places')
    count = 0
    for l in range(N):
        print(f'{l} experiment')
        J = instances_dict[str(grid_size)][l][0]
        I = np.array(instances_dict[str(grid_size)][l][1])
        I = I / max(I)
        S = 1
        P = int(instances_dict[str(grid_size)][l][2] * 0.5)
        EqC = P
        alpha = np.mean(I)
        model = MCLP(J, I, P, S, version= 'COP', slack_vars=slack_vars) 
        n_qubits = model.number_of_variables
        
        backend = AerSimulator()

        t_ws_t = 0
        if WS:
            cont_model = MCLP(J, I, P, S, version= 'CONT', slack_vars=slack_vars)
            t_ws_i = time.time()
            cont_sol = cont_solution(cont_model)
            WS_params = WS_parameters(cont_sol)
            t_ws_f = time.time()
            t_ws_t = t_ws_f-t_ws_i
        else:
            WS_params = None
        
        ising_model = QUBOtoIsingModel(
            model,
            include='Total', 
            normalize=False, 
            obj_normalize=False, 
            pen_normalize=False, 
            UP=not slack_vars,
            alpha=alpha, 
            EqC=EqC,
            gamma=0, 
            precision=6
        )

        opt_sol = solution(model)
        opt_sol_cost = compute_cost({opt_sol:1}, ising_model)

        e_min = 0
        brute_force_list = list(BruteForceSolutions(model, alpha, 0, EqC=EqC, 
                                                    UP=not slack_vars, 
                                                    include='Total', 
                                                    normalize=False, 
                                                    obj_normalize=False, 
                                                    pen_normalize=False).items())
        s_min = brute_force_list[0][0]
        e_min = brute_force_list[0][1]
        e_max = brute_force_list[-1][1]
        h_d = hammingDist(s_min, opt_sol)
        data['hamming_d'].append(h_d)
        data['ham_energies'].append((e_min, e_max))

        r_n = []
        probs_n = []
        qaoa_fes = []
        t_qaoa = []
        t = []
        n_fev = []
        qaoa_e = []
        gates_count = []

        for p in ps:
            t_qaoa_i = time.time()
            res, energies, probs, qaoa_result, qaoa_energy, gates = min_QAOA(model, ising_model, p, backend, LR=LR, WS_params=WS_params, iters=iters)
            qaoa_e.append(qaoa_energy)
            n_fev.append(res.nfev)
            gates_count.append(gates)
            t_qaoa_f = time.time()
            t_qaoa_t = t_qaoa_f - t_qaoa_i
            t_qaoa.append(t_qaoa_t)
            t.append(t_ws_t + t_qaoa_t)
            probs_n.append(qaoa_result.get(opt_sol,0)/shots)
            
            r = (qaoa_energy - e_max) / (e_min - e_max)
            r_n.append(r)
            
            fes = 0
            for s, c in qaoa_result.items():
                fes += check_feasibility(model, s) * c / shots
            qaoa_fes.append(fes)
            print(f"r: {r}, probs: {qaoa_result.get(opt_sol,0)/shots}, t: {t_ws_t + t_qaoa_t}, hamming_distance:{h_d}")
        data['n_qubits'].append(n_qubits)  
        data['qaoa_FR'].append(qaoa_fes)
        data['qaoa_e'].append(qaoa_e)
        data['r'].append(r_n)
        data['probs'].append(probs_n)
        data['gates'].append(gates_count)
        data['t_qaoa'].append(t_qaoa)
        data['t_total'].append(t)
        data['n_fev'].append(n_fev)
    return data

In [7]:
ps = [1, 20, 40, 60, 80, 100]
grid_size = [2, 4]
iterations = 1000
N = 10
alpha = None # alpha is defined inside run_MCLP

data = {}
file_name = f"data_ps_{grid_size[0]}_{grid_size[1]}_{iterations}.txt"

print('UP')
data['UP'] = run_MCLP(grid_size, instances_dict, ps, N, alpha, slack_vars=False, WS=False, LR=False, iters=iterations)
with open(file_name, "w") as file:
    file.write(str(data))
file.close()

print('UP_LR')
data['UP_LR'] = run_MCLP(grid_size, instances_dict, ps, N, alpha, slack_vars=False, WS=False, LR=True, iters=iterations)
with open(file_name, "w") as file:
    file.write(str(data))
file.close()

print('UP_WS')
data['UP_WS'] = run_MCLP(grid_size, instances_dict, ps, N, alpha, slack_vars=False, WS=True, LR=False, iters=iterations)
with open(file_name, "w") as file:
    file.write(str(data))
file.close()

print('UP_WS_LR')
data['UP_WS_LR'] = run_MCLP(grid_size, instances_dict, ps, N, alpha, slack_vars=False, WS=True, LR=True, iters=iterations)
with open(file_name, "w") as file:
    file.write(str(data))
file.close()

UP
[2, 4] places
0 experiment
r: 0.7389567779960708, probs: 0.0, t: 1.3322367668151855, hamming_distance:0
r: 0.7314518664047153, probs: 0.002, t: 36.060340881347656, hamming_distance:0
r: 0.7258074656188606, probs: 0.0, t: 104.24658870697021, hamming_distance:0
r: 0.7302534381139489, probs: 0.0, t: 199.0825138092041, hamming_distance:0
r: 0.731007858546169, probs: 0.0, t: 250.28075742721558, hamming_distance:0
r: 0.7238506876227898, probs: 0.0, t: 304.8754985332489, hamming_distance:0
1 experiment
r: 0.8083179487179488, probs: 0.0, t: 1.5148818492889404, hamming_distance:0
r: 0.7371384615384616, probs: 0.0, t: 38.28820443153381, hamming_distance:0
r: 0.6984974358974358, probs: 0.0, t: 111.12472343444824, hamming_distance:0
r: 0.7518461538461538, probs: 0.0, t: 202.7728283405304, hamming_distance:0
r: 0.7214410256410256, probs: 0.0, t: 257.3416368961334, hamming_distance:0
r: 0.7068102564102564, probs: 0.0, t: 312.04625964164734, hamming_distance:0
2 experiment
r: 0.7246338028169014, p